# YOLOv8 Microplastic Particle Detection Training Guide
This notebook provides the complete pipeline to train the YOLOv8 model on a GPU (e.g. Google Colab Free T4) using public microplastic image datasets, export the fine-tuned weights, and transition AquaScan out of Demo Mode.

### Pipeline Overview:
1. Environment Setup & Dependency Installation
2. Dataset Acquisition (Roboflow / Public Datasets)
3. Dataset Formatting & Configuration
4. YOLOv8 Model Fine-tuning
5. Model Validation & Inference Verification
6. Weight Export for local AquaScan deployment

In [ ]:
# 1. Environment Setup
# Run this in Google Colab (ensure Runtime -> Change runtime type -> T4 GPU is selected)
!pip install ultralytics roboflow opencv-python pillow matplotlib

In [ ]:
# Verify GPU accessibility
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

## 2. Dataset Acquisition from Roboflow
We recommend using a public microplastics dataset. You can obtain a free API key from Roboflow (https://roboflow.com) to download standard datasets in YOLOv8 format.

In [ ]:
from roboflow import Roboflow
# Replace with your personal Roboflow API key and workspace/project ids
# Roboflow provides pre-labeled datasets of microplastic fragments, fibers, pellets, and films.
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("microplastics-workspace").project("river-microplastics")
version = project.version(1)
dataset = version.download("yolov8")

## 3. Dataset Configuration
The dataset download will create a `data.yaml` configuration file. Make sure it maps to the classes used in AquaScan:
- 0: fragment
- 1: fiber
- 2: pellet
- 3: film

In [ ]:
# Example of data.yaml format:
yaml_content = f"""
path: {dataset.location}  # dataset root dir
train: train/images
val: valid/images
test: test/images

names:
  0: fragment
  1: fiber
  2: pellet
  3: film
"""
with open('dataset_colab.yaml', 'w') as f:
    f.write(yaml_content)

## 4. Fine-Tuning YOLOv8 (Optimized for Small Objects)
To detect tiny microplastics well, we apply key optimizations:
1. **Model Backbone**: Choose Small (`yolov8s.pt`) or Medium (`yolov8m.pt`) instead of Nano for more representation capacity.
2. **Higher Resolution**: Train at `imgsz=960` or `imgsz=1280` instead of 640 so small particles do not get downsampled to 0 pixels.
3. **Data Augmentations**: Enable vertical flip (`flipud=0.5`), scaling (`scale=0.6`), rotations (`degrees=15.0`), and shear (`shear=2.0`).
4. **Optimizer**: Use `AdamW` with adjusted learning rate for better convergence on sparse datasets.

In [ ]:
from ultralytics import YOLO

# Load a pre-trained small model (s) or medium model (m) for higher detection accuracy
model = YOLO('yolov8s.pt')  # Can also use 'yolov8m.pt' for even higher recall if GPU memory allows

# Start training with optimized parameters
results = model.train(
    data='dataset_colab.yaml',
    epochs=150,               # More epochs give the model time to converge
    imgsz=960,                # High resolution is crucial for small microplastic particles
    batch=8,                  # Reduced batch size (from 16 to 8) to avoid CUDA Out-Of-Memory at imgsz=960
    device=0,                 # GPU
    workers=2,
    optimizer='AdamW',        # AdamW is robust for sparse object detection
    lr0=0.001,                # Learning rate tuning
    lrf=0.01,
    patience=30,              # Early stopping patience to avoid overfitting
    # Data Augmentation parameters
    degrees=15.0,             # Rotate images randomly by up to 15 degrees
    scale=0.6,                # Scale images randomly by up to +/- 60%
    shear=2.0,                # Shear transformation
    flipud=0.5,               # Vertical flips are natural for river-surface images
    fliplr=0.5,               # Horizontal flips
    mixup=0.1,                # Mix two images to regularize training
    project='aquascan_project',
    name='yolov8_microplastics_advanced'
)

## 5. Validation and Testing (with TTA)
Evaluate the model's Mean Average Precision (mAP) on the validation dataset split. We can enable Test-Time Augmentation (TTA) using `augment=True` for maximum detection accuracy.

In [ ]:
# Validate the model (enabling augment=True runs Test-Time Augmentation to detect small particles)
metrics = model.val(augment=True)
print(f"TTA Validation mAP50-95: {metrics.box.map:.4f}")
print(f"TTA Validation mAP50: {metrics.box.map50:.4f}")

In [ ]:
# Run inference on a test image to visually confirm bounding box coordinates
import cv2
from PIL import Image
import matplotlib.pyplot as plt

test_image_path = "test_sample.jpg" # replace with an uploaded sample
results = model(test_image_path)
annotated_frame = results[0].plot()

# Plot result
plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

## 6. How to Export and Deploy locally in AquaScan

Once training completes, Google Colab saves the optimal weights file at:
`aquascan_project/yolov8_microplastics/weights/best.pt`

### Steps to Deploy:
1. Download `best.pt` from the Google Colab file manager to your local computer.
2. Move/copy the downloaded `best.pt` file into the AquaScan project weights directory: `models/weights/best.pt`.
3. Restart the AquaScan Streamlit server.
4. The yellow warning banner will disappear, and the system will run **real, accurate AI inference** on all uploaded images!